# 0.0 - Imports

In [1]:
import requests
import pandas as pd
import time
import os
import sqlite3
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

### 0.1 - Helper Functions

In [2]:
class LimitadorDeTaxa:
    """
    Garante um espaçamento mínimo GLOBAL entre requisições, mesmo com
    múltiplas threads rodando ao mesmo tempo. Sem isso, o ThreadPoolExecutor
    dispara várias requisições simultâneas e estoura o rate limit da API
    (é o que causou os 429 com espera de 59s na tentativa anterior).

    Para ajustar o ritmo: mude apenas o valor de INTERVALO_MINIMO_SEGUNDOS
    logo abaixo desta classe — é o único lugar que precisa mudar.
    """
    def __init__(self, intervalo_minimo=0.7):
        self.intervalo_minimo = intervalo_minimo
        self.lock = threading.Lock()
        self.ultima_chamada = 0.0

    def aguardar(self):
        with self.lock:
            agora = time.time()
            espera = self.intervalo_minimo - (agora - self.ultima_chamada)
            if espera > 0:
                time.sleep(espera)
            self.ultima_chamada = time.time()


# ── AJUSTE AQUI se precisar mudar o ritmo das requisições ──────────────────
INTERVALO_MINIMO_SEGUNDOS = 0.7  # comprovadamente seguro na 1ª execução (sequencial)
limitador = LimitadorDeTaxa(intervalo_minimo=INTERVALO_MINIMO_SEGUNDOS)


def requisicao_com_retry(url, params, tentativas=4):
    """
    Faz uma requisição GET com tentativas repetidas em caso de falha.
    Passa sempre pelo limitador de taxa global antes de disparar a
    requisição, e respeita o header 'Retry-After' quando a API devolve 429.

    Retorna
    -------
    dados : dict ou None
        Corpo da resposta JSON, ou None se todas as tentativas falharem.
    """
    for tentativa in range(1, tentativas + 1):
        limitador.aguardar()
        try:
            resposta = requests.get(url, params=params, timeout=30)
            if resposta.status_code == 200:
                return resposta.json()

            if resposta.status_code == 429:
                espera = int(resposta.headers.get("Retry-After", 2 * tentativa))
                print(f"  [429] aguardando {espera}s em {params} "
                      f"(tentativa {tentativa}/{tentativas})")
                time.sleep(espera)
                continue

            print(f"  [aviso] status {resposta.status_code} em {params} "
                  f"(tentativa {tentativa}/{tentativas})")
        except requests.exceptions.RequestException as erro:
            print(f"  [aviso] erro de conexão: {erro} (tentativa {tentativa}/{tentativas})")

        time.sleep(2 * tentativa)

    print(f"  [FALHOU] desistindo após {tentativas} tentativas: {params}")
    return None


def coletar_e_salvar_material(codigo_grupo, codigo_classe, conexao):
    """
    Coleta itens de material de uma classe e insere direto em dim_material,
    página a página, ignorando duplicatas automaticamente.

    Retorna
    -------
    total_coletado : int
        Quantidade de itens processados nesta classe.
    """
    pagina = 1
    total_coletado = 0

    while True:
        params = {
            "codigoGrupo": codigo_grupo,
            "codigoClasse": codigo_classe,
            "pagina": pagina,
            "tamanhoPagina": 500,
        }
        dados = requisicao_com_retry(URL_ITEM_MATERIAL, params)
        if dados is None or not dados["resultado"]:
            break

        df_pagina = pd.DataFrame(dados["resultado"]).rename(columns={
            "codigoItem": "codigo_item",
            "codigoClasse": "codigo_classe",
            "nomeClasse": "nome_classe",
            "codigoGrupo": "codigo_grupo",
            "nomeGrupo": "nome_grupo",
            "codigoPdm": "codigo_pdm",
            "nomePdm": "nome_pdm",
            "descricaoItem": "descricao_item",
        })[["codigo_item", "codigo_classe", "nome_classe", "codigo_grupo",
            "nome_grupo", "codigo_pdm", "nome_pdm", "descricao_item"]]

        cursor = conexao.cursor()
        cursor.executemany(
            """INSERT OR IGNORE INTO dim_material
                (codigo_item, codigo_classe, nome_classe, codigo_grupo,
                 nome_grupo, codigo_pdm, nome_pdm, descricao_item)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?)""",
            df_pagina.itertuples(index=False, name=None)
        )
        conexao.commit()

        total_coletado += len(df_pagina)
        total_paginas = dados.get("totalPaginas", 1)

        if pagina >= total_paginas:
            break
        pagina += 1
        time.sleep(PAUSA_ENTRE_CHAMADAS)

    return total_coletado


def itens_ja_coletados(conexao):
    """
    Retorna o conjunto de codigo_item_catalogo que já possuem registros
    na tabela fato_precos_praticados — usado para pular no resume.

    Retorna
    -------
    codigos_existentes : set[int]
        Códigos de item já processados em uma execução anterior.
    """
    df = pd.read_sql(
        "SELECT DISTINCT codigo_item_catalogo FROM fato_precos_praticados",
        conexao
    )
    return set(df["codigo_item_catalogo"].tolist())


def buscar_precos_item(codigo_item_catalogo):
    """
    Busca (sem gravar) todas as páginas de preços de um item.
    Feita para rodar em paralelo (ThreadPoolExecutor) — só faz requisição
    HTTP, não toca no banco, então múltiplas chamadas simultâneas são seguras.

    Retorna
    -------
    codigo_item_catalogo : int
        O mesmo código recebido (útil para rastrear qual future é qual).
    registros : list[dict]
        Todos os registros de preço coletados para esse item.
    """
    registros = []
    pagina = 1

    while True:
        params = {
            "codigoItemCatalogo": codigo_item_catalogo,
            "pagina": pagina,
            "tamanhoPagina": 500,
        }
        dados = requisicao_com_retry(URL_PESQUISA_PRECO, params)
        if dados is None or not dados["resultado"]:
            break

        registros.extend(dados["resultado"])
        total_paginas = dados.get("totalPaginas", 1)
        if pagina >= total_paginas:
            break
        pagina += 1
        time.sleep(PAUSA_ENTRE_CHAMADAS)

    return codigo_item_catalogo, registros


def salvar_precos_no_banco(registros, conexao):
    """
    Grava no SQLite os registros de preço já buscados. Deve ser chamada
    sempre a partir da thread principal (nunca dentro do ThreadPoolExecutor),
    para não gerar concorrência de escrita no SQLite.

    Parâmetros
    ----------
    registros : list[dict]
        Registros retornados por buscar_precos_item.
    conexao : sqlite3.Connection
        Conexão ativa com o banco.

    Retorna
    -------
    total_gravado : int
        Quantidade de linhas inseridas em fato_precos_praticados.
    """
    if not registros:
        return 0

    df_pagina = pd.DataFrame(registros)
    cursor = conexao.cursor()

    # ── Dimensão fornecedor ──────────────────────────────────────────────
    df_fornecedor = df_pagina[["niFornecedor", "nomeFornecedor"]].dropna(
        subset=["niFornecedor"]).drop_duplicates(subset=["niFornecedor"])
    cursor.executemany(
        "INSERT OR IGNORE INTO dim_fornecedor (ni_fornecedor, nome_fornecedor) VALUES (?, ?)",
        df_fornecedor.itertuples(index=False, name=None))

    # ── Dimensão UASG ────────────────────────────────────────────────────
    df_uasg = df_pagina[["codigoUasg", "nomeUasg", "codigoMunicipio", "municipio",
                          "estado", "codigoOrgao", "nomeOrgao", "poder", "esfera"]].dropna(
        subset=["codigoUasg"]).drop_duplicates(subset=["codigoUasg"])
    cursor.executemany(
        """INSERT OR IGNORE INTO dim_uasg
           (codigo_uasg, nome_uasg, codigo_municipio, municipio, estado,
            codigo_orgao, nome_orgao, poder, esfera)
           VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)""",
        df_uasg.itertuples(index=False, name=None))

    # ── Fato ─────────────────────────────────────────────────────────────
    # Nota: idCompraItem (string) é o identificador correto — confirmado
    # no Schema real do Swagger. idItemCompra (int) é só o número
    # sequencial do item dentro da compra, não um identificador único.
    df_fato = df_pagina.rename(columns={
        "idCompraItem": "id_compra_item", "codigoItemCatalogo": "codigo_item_catalogo",
        "niFornecedor": "ni_fornecedor", "codigoUasg": "codigo_uasg",
        "precoUnitario": "preco_unitario", "dataCompra": "data_compra",
        "dataResultado": "data_resultado", "siglaUnidadeMedida": "sigla_unidade_medida",
        "nomeUnidadeMedida": "nome_unidade_medida", "criterioJulgamento": "criterio_julgamento",
        "objetoCompra": "objeto_compra", "codigoClasse": "codigo_classe",
        "nomeClasse": "nome_classe", "dataHoraAtualizacaoItem": "data_hora_atualizacao_item",
    })[["id_compra_item", "codigo_item_catalogo", "ni_fornecedor", "codigo_uasg",
        "quantidade", "preco_unitario", "data_compra", "data_resultado", "marca",
        "sigla_unidade_medida", "nome_unidade_medida", "criterio_julgamento",
        "modalidade", "objeto_compra", "codigo_classe", "nome_classe",
        "data_hora_atualizacao_item"]]

    cursor.executemany(
    """INSERT OR IGNORE INTO fato_precos_praticados
           (id_compra_item, codigo_item_catalogo, ni_fornecedor, codigo_uasg,
            quantidade, preco_unitario, data_compra, data_resultado, marca,
            sigla_unidade_medida, nome_unidade_medida, criterio_julgamento,
            modalidade, objeto_compra, codigo_classe, nome_classe, data_hora_atualizacao_item)
           VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)""",
        df_fato.itertuples(index=False, name=None))

    conexao.commit()
    return len(df_fato)

### 0.2 - Caminho do Arquivo

In [3]:
PASTA_SQL = "../sql"
CAMINHO_BANCO = "../assets/data/database.db"

os.makedirs(PASTA_SQL, exist_ok=True)
os.makedirs("../assets/data", exist_ok=True)

MAX_WORKERS = 3  # threads simultâneas buscando na API (ajuste se houver muito 429)

print("Pastas prontas.")

Pastas prontas.


### 1.0 - Criação de Grupo e Sub-Grupo de Materiais

In [4]:
#1. Criação de Dicionário para requisição de materiais (grupo e classes)

## 61 - CONDUTORES ELÉTRICOS E EQUIPAMENTOS PARA GERAÇÃO E DISTRIBUIÇÃO DE ENERGIA
### 6120 - Transformadores para estação de força e de distribuição
### 6145 - Fios e cabos elétricos
### 6150 - Equipamentos diversos p/ geração e distribuição de energia

## 59 - COMPONENTES ELETRICOS - COMPLEMENTAR
### 5925 - Disjuntores
### 5930 - Chaves elétricas
### 5970 - Isoladores elétricos e materiais isolantes
### 5975 - Ferragens e suprimentos de eletricidade

CLASSES_MATERIAL_ELETRICO = [
    (61, 6120),
    (61, 6145),
    (61, 6150),
    (59, 5925),
    (59, 5930),
    (59, 5970),
    (59, 5975),
]

URL_ITEM_MATERIAL = "https://dadosabertos.compras.gov.br/modulo-material/4_consultarItemMaterial"
URL_PESQUISA_PRECO = "https://dadosabertos.compras.gov.br/modulo-pesquisa-preco/1_consultarMaterial"

PAUSA_ENTRE_CHAMADAS = 0.4

### 2.0 - Criação do Schema do Banco de Dados

In [5]:
# Schema do Banco de Dados SQL

schema_sql = """
CREATE TABLE IF NOT EXISTS dim_material (
    codigo_item     INTEGER PRIMARY KEY,
    codigo_classe   INTEGER,
    nome_classe     TEXT,
    codigo_grupo    INTEGER,
    nome_grupo      TEXT,
    codigo_pdm      INTEGER,
    nome_pdm        TEXT,
    descricao_item  TEXT
);

CREATE TABLE IF NOT EXISTS dim_fornecedor (
    ni_fornecedor   TEXT PRIMARY KEY,
    nome_fornecedor TEXT
);

CREATE TABLE IF NOT EXISTS dim_uasg (
    codigo_uasg      TEXT PRIMARY KEY,
    nome_uasg        TEXT,
    codigo_municipio INTEGER,
    municipio        TEXT,
    estado           TEXT,
    codigo_orgao     INTEGER,
    nome_orgao       TEXT,
    poder            TEXT,
    esfera           TEXT
);

CREATE TABLE IF NOT EXISTS fato_precos_praticados (
    id_compra_item              TEXT,
    codigo_item_catalogo        INTEGER,
    ni_fornecedor                TEXT,
    codigo_uasg                  TEXT,
    quantidade                   REAL,
    preco_unitario               REAL,
    data_compra                  TEXT,
    data_resultado               TEXT,
    marca                        TEXT,
    sigla_unidade_medida         TEXT,
    nome_unidade_medida          TEXT,
    criterio_julgamento          TEXT,
    modalidade                   INTEGER,
    objeto_compra                TEXT,
    codigo_classe                INTEGER,
    nome_classe                  TEXT,
    data_hora_atualizacao_item   TEXT,
    FOREIGN KEY (codigo_item_catalogo) REFERENCES dim_material (codigo_item),
    FOREIGN KEY (ni_fornecedor) REFERENCES dim_fornecedor (ni_fornecedor),
    FOREIGN KEY (codigo_uasg) REFERENCES dim_uasg (codigo_uasg)
);
"""

In [6]:
#Caminho do Schema para Salvar API -> Banco de Dados
caminho_schema = os.path.join(PASTA_SQL, "01_criar_schema.sql")
with open(caminho_schema, "w", encoding="utf-8") as f:
    f.write(schema_sql)

conexao = sqlite3.connect(CAMINHO_BANCO)
conexao.executescript(schema_sql)
conexao.commit()

print(f"Schema salvo em {caminho_schema}")
print(f"Banco pronto em {CAMINHO_BANCO}")

Schema salvo em ../sql\01_criar_schema.sql
Banco pronto em ../assets/data/database.db


### 3.0 - Chamada API - Coleta

In [7]:
# Busca na Criação de Grupos e Sub Grupos de Materiais e aplica a função coletar_e_salvar_materiais (helper_functions)
for codigo_grupo, codigo_classe in CLASSES_MATERIAL_ELETRICO:
    print(f"Classe {codigo_classe} (grupo {codigo_grupo})...")
    total = coletar_e_salvar_material(codigo_grupo, codigo_classe, conexao)
    print(f"  → {total} itens processados (novos ou já existentes)")
    time.sleep(PAUSA_ENTRE_CHAMADAS)

total_dim_material = pd.read_sql("SELECT COUNT(*) AS total FROM dim_material", conexao)
print(f"\nTotal em dim_material: {total_dim_material['total'][0]}")

Classe 6120 (grupo 61)...
  → 130 itens processados (novos ou já existentes)
Classe 6145 (grupo 61)...
  → 3256 itens processados (novos ou já existentes)
Classe 6150 (grupo 61)...
  → 285 itens processados (novos ou já existentes)
Classe 5925 (grupo 59)...
  → 1604 itens processados (novos ou já existentes)
Classe 5930 (grupo 59)...
  → 1298 itens processados (novos ou já existentes)
Classe 5970 (grupo 59)...
  → 548 itens processados (novos ou já existentes)
Classe 5975 (grupo 59)...
  → 3795 itens processados (novos ou já existentes)

Total em dim_material: 10916


In [8]:
#Loop principal — com a checagem de resume logo no início:
codigos_item_material = pd.read_sql(
    "SELECT DISTINCT codigo_item FROM dim_material", conexao
)["codigo_item"].tolist()

ja_coletados = itens_ja_coletados(conexao)
pendentes = [c for c in codigos_item_material if c not in ja_coletados]

print(f"Total de itens de material: {len(codigos_item_material)}")
print(f"Já coletados em execução anterior: {len(ja_coletados)}")
print(f"Pendentes nesta execução: {len(pendentes)}")

Total de itens de material: 10916
Já coletados em execução anterior: 1246
Pendentes nesta execução: 9670


In [9]:
#Executa a coleta de preços em paralelo (busca) + gravação sequencial (SQLite)
#Se travar no meio, é só rodar a célula de "resume" (acima) de novo e depois esta célula de novo:
#os itens já gravados são pulados automaticamente, sem gastar requisição.

total_gravado = 0

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(buscar_precos_item, c): c for c in pendentes}

    for i, future in enumerate(as_completed(futures), start=1):
        codigo_item, registros = future.result()
        gravados = salvar_precos_no_banco(registros, conexao)
        total_gravado += gravados

        if i % 50 == 0 or i == 1:
            print(f"Progresso: {i}/{len(pendentes)} itens processados "
                  f"({total_gravado} registros gravados até agora)")

print("\nColeta de preços concluída (ou retomada com sucesso).")

total_fato = pd.read_sql("SELECT COUNT(*) AS total FROM fato_precos_praticados", conexao)
print(f"Total de registros em fato_precos_praticados: {total_fato['total'][0]}")

Progresso: 1/9670 itens processados (0 registros gravados até agora)
Progresso: 50/9670 itens processados (0 registros gravados até agora)
Progresso: 100/9670 itens processados (0 registros gravados até agora)
Progresso: 150/9670 itens processados (0 registros gravados até agora)
Progresso: 200/9670 itens processados (0 registros gravados até agora)
Progresso: 250/9670 itens processados (0 registros gravados até agora)
Progresso: 300/9670 itens processados (0 registros gravados até agora)
Progresso: 350/9670 itens processados (0 registros gravados até agora)
Progresso: 400/9670 itens processados (0 registros gravados até agora)
Progresso: 450/9670 itens processados (0 registros gravados até agora)
Progresso: 500/9670 itens processados (0 registros gravados até agora)
Progresso: 550/9670 itens processados (0 registros gravados até agora)
Progresso: 600/9670 itens processados (0 registros gravados até agora)
Progresso: 650/9670 itens processados (0 registros gravados até agora)
Progresso

### 4.0 - Validação SQL

In [10]:
queries_validacao = """
-- 1. Contagem geral por tabela
SELECT 'dim_material' AS tabela, COUNT(*) AS total FROM dim_material
UNION ALL
SELECT 'dim_fornecedor', COUNT(*) FROM dim_fornecedor
UNION ALL
SELECT 'dim_uasg', COUNT(*) FROM dim_uasg
UNION ALL
SELECT 'fato_precos_praticados', COUNT(*) FROM fato_precos_praticados;

-- 2. Registros do fato sem correspondência em dim_material (órfãos)
SELECT COUNT(*) AS itens_orfaos
FROM fato_precos_praticados f
LEFT JOIN dim_material m ON f.codigo_item_catalogo = m.codigo_item
WHERE m.codigo_item IS NULL;

-- 3. Valores nulos ou zerados em campos críticos
SELECT
    SUM(CASE WHEN preco_unitario IS NULL OR preco_unitario <= 0 THEN 1 ELSE 0 END) AS preco_invalido,
    SUM(CASE WHEN quantidade IS NULL OR quantidade <= 0 THEN 1 ELSE 0 END) AS quantidade_invalida,
    SUM(CASE WHEN ni_fornecedor IS NULL THEN 1 ELSE 0 END) AS fornecedor_nulo
FROM fato_precos_praticados;

-- 4. Top 10 materiais por volume de registros
SELECT m.nome_classe, m.descricao_item, COUNT(*) AS qtd_registros
FROM fato_precos_praticados f
JOIN dim_material m ON f.codigo_item_catalogo = m.codigo_item
GROUP BY m.codigo_item
ORDER BY qtd_registros DESC
LIMIT 10;
"""

caminho_validacao = os.path.join(PASTA_SQL, "02_queries_validacao.sql")
with open(caminho_validacao, "w", encoding="utf-8") as f:
    f.write(queries_validacao)

print(f"Queries salvas em {caminho_validacao}")

Queries salvas em ../sql\02_queries_validacao.sql


In [12]:
pd.read_sql("""
    SELECT 'dim_material' AS tabela, COUNT(*) AS total FROM dim_material
    UNION ALL
    SELECT 'dim_fornecedor', COUNT(*) FROM dim_fornecedor
    UNION ALL
    SELECT 'dim_uasg', COUNT(*) FROM dim_uasg
    UNION ALL
    SELECT 'fato_precos_praticados', COUNT(*) FROM fato_precos_praticados
""", conexao)

,tabela,total
0,dim_material,10916
1,dim_fornecedor,10235
2,dim_uasg,3687
3,fato_precos_praticados,164680


In [13]:
pd.read_sql("""
    SELECT COUNT(*) AS itens_orfaos
    FROM fato_precos_praticados f
    LEFT JOIN dim_material m ON f.codigo_item_catalogo = m.codigo_item
    WHERE m.codigo_item IS NULL
""", conexao)

,itens_orfaos
0,0


In [14]:
pd.read_sql("""
    SELECT
        SUM(CASE WHEN preco_unitario IS NULL OR preco_unitario <= 0 THEN 1 ELSE 0 END) AS preco_invalido,
        SUM(CASE WHEN quantidade IS NULL OR quantidade <= 0 THEN 1 ELSE 0 END) AS quantidade_invalida,
        SUM(CASE WHEN ni_fornecedor IS NULL THEN 1 ELSE 0 END) AS fornecedor_nulo
    FROM fato_precos_praticados
""", conexao)

,preco_invalido,quantidade_invalida,fornecedor_nulo
0,0,0,0


In [15]:
pd.read_sql("""
    SELECT m.nome_classe, m.descricao_item, COUNT(*) AS qtd_registros
    FROM fato_precos_praticados f
    JOIN dim_material m ON f.codigo_item_catalogo = m.codigo_item
    GROUP BY m.codigo_item
    ORDER BY qtd_registros DESC
    LIMIT 10
""", conexao)

,nome_classe,descricao_item,qtd_registros
0,ISOLADORES ELÉTRICOS E MATERIAIS ISOLANTES,"FITA ISOLANTE ELÉTRICA, MATERIAL BÁSICO: FILME...",1415
1,DISJUNTORES,"PEÇA / ACESSÓRIO DISJUNTOR, NOME: PEÇA / ACESS...",853
2,ISOLADORES ELÉTRICOS E MATERIAIS ISOLANTES,"FITA ISOLANTE ELÉTRICA ADESIVA, MATERIAL DORSO...",719
3,FIOS E CABOS ELÉTRICOS,"CABO ÁUDIO E VÍDEO, MATERIAL CONDUTOR: COBRE ,...",713
4,FIOS E CABOS ELÉTRICOS,"CABO ÁUDIO E VÍDEO, APLICAÇÃO: SISTEMA DE ÁUDI...",574
5,ISOLADORES ELÉTRICOS E MATERIAIS ISOLANTES,"FITA ISOLANTE ELÉTRICA, MATERIAL BÁSICO: FILME...",519
6,ISOLADORES ELÉTRICOS E MATERIAIS ISOLANTES,"FITA ISOLANTE ELÉTRICA ADESIVA, MATERIAL DORSO...",495
7,DISJUNTORES,"DISJUNTOR BAIXA TENSÃO, FUNCIONAMENTO: TERMOMA...",492
8,CHAVES ELÉTRICAS,"INTERRUPTOR, TIPO: PARALELO_(THREE-WAY) , QUAN...",489
9,DISJUNTORES,"DISJUNTOR BAIXA TENSÃO, FUNCIONAMENTO: TERMOMA...",488


In [16]:
#Fechamento da Conexão
conexao.close()
print("Conexão fechada. Banco salvo em:", CAMINHO_BANCO)

Conexão fechada. Banco salvo em: ../assets/data/database.db
